# GRPO fine-tune of Phi-4-mini on the pharma cold-chain environment (Kaggle)

**Goal: a first positive signal.** The trained model should read the news better than the untrained one,
measured on seeds it never saw.

**What the model learns.** The `adjust` format: on each day the baseline rule places a default order, and the model answers
only with **extra safety days per product (0 to +10)**. Training uses only **news days** from the **random scenario**
with **level-2 news**.

**Two stages** (`WARM_START = True`, the default):
1. **Warm start, supervised.** The model first learns the answers of a **teacher** (`training/teacher.py`): an inventory plan
   built from the exact events behind the news, answering in the same format. Playing those answers scores 0.887 on test
   seeds (baseline 0.859, "+3 everywhere" 0.869, oracle 0.925), so a model that imitates it well is already useful.
2. **GRPO.** Then the model improves on its own answers with the environment's reward.

With `WARM_START = False` the notebook runs GRPO alone, as before.

**Reward** (`training/grpo_reward.py`, spec in `training/REWARD.md`): the answer is applied today, and then the teacher
(the baseline without the warm start) plays 15 days in each of 4 shared possible futures. The reward is the gain over answering 0, in points
(×100/15). Format penalties are −1 for no JSON and −0.25 for a value out of range. Each prompt gets a group of
8 answers, with Dr. GRPO (`scale_rewards=False`).

**What the notebook does**

| Part | What happens |
|---|---|
| 0–2 | Setup, config, model + LoRA |
| 3 | Training prompts (train seeds 0–8999) and the probe exam (held-out seeds 9000–9099) |
| 4 | **Before training:** test-seed probe and full episodes with the untrained model |
| 4b | **Warm start:** supervised fine-tuning on the teacher's answers, then the probe exam again |
| 5–6 | GRPO training with health metrics, and a probe exam every 25 steps |
| 7 | Save the LoRA adapter |
| 8 | **After training:** test-seed probe, full episodes, news-0 and fixed-scenario checks, the verdict |
| 9 | Qualitative look, GGUF export for Ollama, and a results zip |

**Kaggle settings.**
- **Accelerator:** GPU T4 ×1 (or P100).
- **Internet:** on.
- **Run mode:** *Save Version → Save & Run All* (runs in the background, up to 12 h).
- **Time:** about 6–8 h on a T4 for GRPO and the evaluations, plus the warm start (not yet timed on a T4: expect
  roughly an hour for the default `SFT_EXAMPLES`; part 4b prints its own time). Lower `TRAIN_STEPS` / `EVAL_SEEDS` /
  `SFT_EXAMPLES` for a quicker first run.
  Part 4 prints the measured time per answer so you can extrapolate.

Everything is written to `/kaggle/working/` (shown under *Output*).

## 0. Setup: install, get the code

In [ ]:
# Unsloth brings a matching torch / transformers / TRL / PEFT. openenv-core is the environment's only other dependency.
!pip install -q unsloth
!pip install -q "openenv-core>=0.2.1" plotly

In [ ]:
import os, subprocess, sys

REPO = "BishnuGanguly/pharma_coldstorage_env"
BRANCH = "feature/scenario-config"   # has the teacher; switch to "main" once that branch is merged
WORK = "/kaggle/working"
CODE_DIR = "/kaggle/working/pharma_coldstorage_env"

# A private repo needs a GitHub token saved as the Kaggle secret GITHUB_TOKEN (Add-ons -> Secrets).
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
if not os.path.isdir(CODE_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, CODE_DIR], check=True)
print(subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)

In [ ]:
# Unsloth must be imported before transformers / TRL.
import unsloth  # noqa: F401
from unsloth import FastLanguageModel, is_bfloat16_supported

import json, math, random, statistics as st, time
from collections import deque
from dataclasses import fields
from pathlib import Path

import torch
from transformers import Trainer, TrainerCallback, TrainingArguments
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

import eval as E
from agents import BaselineAgent
from tasks import TASK_REGISTRY, TEST_SEEDS
from training import episodes as EP
from training import probe as P
from training import teacher as TEACH
from training.grpo_reward import GRPOReward, build_state, dataset_rows, news_day_prompts
from training.monitor import MonitoredReward

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
import trl, transformers, peft
print("trl", trl.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)

## 1. Config

Every setting is in one place. The defaults follow what we measured: temperature 1.0 gave enough variety in the
diversity check; groups of 8; 15-day horizon; 4 futures.

In [ ]:
OUT = Path(WORK) / "grpo_run"
OUT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "unsloth/Phi-4-mini-instruct"
MAX_PROMPT_TOKENS = 3072        # the prompts are ~2,000-2,600 tokens (report + 5-day history)
MAX_COMPLETION_TOKENS = 96      # a JSON object of 5 numbers needs ~40-60
LORA_RANK = 16

TASKS = sorted(TASK_REGISTRY)                 # supply_chain_broken, flu_season, epidemic_two_wave
TRAIN_SEED_POOL = range(0, 9000)              # train split, minus the probe seeds 9000-9099
TRAIN_EPISODES_PER_TASK = 120                 # episodes sampled to make the prompt pool (~14 news days each)
MIN_SPREAD = 0.25    # keep news days where the reference answers (0/2/4/7/10) differ by at least this many points

# Warm start: supervised fine-tuning on the teacher's answers before GRPO (training/teacher.py).
# False runs GRPO alone, exactly as before the teacher existed.
WARM_START = True
SFT_EPISODES_PER_TASK = 40          # episodes the examples come from (~45 examples each)
SFT_EXAMPLES = 1600                 # examples trained on, one pass
SFT_ACCUM = 8                       # examples per optimizer step -> 200 steps
SFT_LEARNING_RATE = 1e-4
EVAL_AFTER_SFT = True               # full episodes after the warm start too (as long as the "before" episodes)
# Who plays the days around the scored answer. After a warm start the model plays like the
# teacher, so its prompts come from the teacher's play and an answer is judged with the
# teacher ordering on the following days (REWARD.md, "Who plays after today").
PLAYER = "teacher" if WARM_START else "baseline"

# GRPO
NUM_GENERATIONS = 8                 # answers per prompt (the group)
PER_DEVICE_BATCH = 8                # completions per forward pass: one group
GRAD_ACCUM = 2                      # -> 2 prompts (16 answers) per optimizer step
TRAIN_STEPS = 300                   # 600 prompts, 4,800 scored answers
LEARNING_RATE = 1e-5
# TRL measures the KL penalty against the BASE model (the LoRA adapter switched off). After a
# warm start that would pull the model back to where it was before it learned the teacher.
BETA_KL = 0.0 if WARM_START else 0.04
TEMPERATURE = 1.0
SAVE_EVERY = 50

# Monitoring and evaluation
PROBE_EVERY = 25                    # steps between probe exams
PROBE_COUNT = 100                   # news days in the probe exam
EVAL_SEEDS = 10                     # test seeds per task for full 60-day episodes (3 tasks -> 30 episodes)
CHECK_SEEDS = 5                     # test seeds per task for the news-0 and fixed-scenario checks
GEN_BATCH = 8                       # prompts per generate() call at evaluation (lower it if out of memory)
EXPORT_GGUF = True

random.seed(0)
torch.manual_seed(0)

## 2. Model: Phi-4-mini in 4-bit, with LoRA

`generate()` is used by the probe exam and the full episodes. It answers greedily, so the base and trained models get
compared on exactly the same prompts with no sampling noise. GRPO samples on its own at `TEMPERATURE`.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_PROMPT_TOKENS + MAX_COMPLETION_TOKENS,
    load_in_4bit=True,
    dtype=None,
)

# Phi-4-mini uses fused qkv_proj / gate_up_proj; other builds split them. Target whichever exist.
module_names = {name.split(".")[-1] for name, _ in model.named_modules()}
candidates = ["qkv_proj", "q_proj", "k_proj", "v_proj", "o_proj", "gate_up_proj", "gate_proj", "up_proj", "down_proj"]
targets = [m for m in candidates if m in module_names]
print("LoRA targets:", targets)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    target_modules=targets,
    use_gradient_checkpointing="unsloth",
    random_state=0,
)
model.print_trainable_parameters()

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def generate(batch_messages, temperature=0.0, max_new_tokens=MAX_COMPLETION_TOKENS, batch_size=GEN_BATCH):
    """One answer per chat (a list of messages). Greedy by default."""
    FastLanguageModel.for_inference(model)
    answers = []
    for i in range(0, len(batch_messages), batch_size):
        chunk = batch_messages[i:i + batch_size]
        texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in chunk]
        enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        sampling = dict(do_sample=True, temperature=temperature, top_p=1.0) if temperature > 0 else dict(do_sample=False)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id, **sampling)
        answers += tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [a.strip() for a in answers]


def back_to_training():
    FastLanguageModel.for_training(model)

## 3. Prompts: training pool and probe exam

- **Training prompts:** news days from train seeds 0–8999, random scenario, news level 2, from episodes played by
  `PLAYER` (the teacher after a warm start). Days where no answer moves the
  reward (reference spread below `MIN_SPREAD`) are dropped, because every answer would score the same. Days where
  *holding back* is right are kept.
- **Probe exam:** 100 news days from seeds 9000–9099, which training never sees. Before training, the probe exam is
  taken again on **test seeds**, so there is a clean before/after on seeds nobody tuned on.

In [ ]:
t0 = time.time()
reward_ref = GRPOReward(continuation=PLAYER)
train_seeds = sorted(random.Random("train-seeds").sample(list(TRAIN_SEED_POOL), TRAIN_EPISODES_PER_TASK))
pool = news_day_prompts(TASKS, train_seeds, states=PLAYER)
random.Random("train-order").shuffle(pool)

needed = TRAIN_STEPS * GRAD_ACCUM * PER_DEVICE_BATCH // NUM_GENERATIONS
kept, dropped = [], 0
for p in pool:
    refs = P.reference_rewards(reward_ref, p)
    if max(refs.values()) - min(refs.values()) >= MIN_SPREAD:
        kept.append(p)
    else:
        dropped += 1
    if len(kept) >= int(needed * 1.2) + 50:
        break
print(f"{len(pool)} news days in the pool; kept {len(kept)} (dropped {dropped} where the answer hardly matters); "
      f"training needs {needed}. ({time.time() - t0:.0f}s)")
print("by task:", {t: sum(p.task == t for p in kept) for t in TASKS})

train_dataset = Dataset.from_list(dataset_rows(kept))
print(train_dataset)

# Prompt length check: GRPO cuts prompts longer than MAX_PROMPT_TOKENS, which would drop the news.
lengths = [len(tokenizer.apply_chat_template(r["prompt"], tokenize=True, add_generation_prompt=True))
           for r in train_dataset.select(range(min(200, len(train_dataset))))]
print(f"prompt tokens: min {min(lengths)}, mean {st.mean(lengths):.0f}, max {max(lengths)} (limit {MAX_PROMPT_TOKENS})")
assert max(lengths) <= MAX_PROMPT_TOKENS, "raise MAX_PROMPT_TOKENS"

In [ ]:
t0 = time.time()
# Both exams use the training reward (reward_ref) and states from the same player as the training prompts.
probe_items = P.build_probe(seeds=P.PROBE_SEEDS, count=PROBE_COUNT, reward=reward_ref, states=PLAYER)   # during training
test_probe_items = P.build_probe(seeds=TEST_SEEDS[:40], count=PROBE_COUNT, salt="test-probe",
                                 reward=reward_ref, states=PLAYER)                                        # before/after
print(f"probe exams built in {time.time() - t0:.0f}s")
for name, items in (("probe (9000-9099)", probe_items), ("test probe", test_probe_items)):
    zero_best = sum(i.zero_is_best for i in items)
    print(f"{name}: {len(items)} days, mean best achievable {st.mean(i.best for i in items):.2f} points, "
          f"holding back is best on {zero_best}")


def run_probe(items, label):
    t = time.time()
    texts = generate([i.messages for i in items])
    results = P.score_probe(items, texts, reward=reward_ref)
    summary = P.summarize_probe(items, results)
    print(f"[{label}] reward {summary['reward']:+.3f} | {summary['pct_of_best']:5.1f}% of best | "
          f"no JSON {summary['no_json']:.0%} | all-zero {summary['all_zero']:.0%} | right product "
          f"{summary['right_product']:.0%} | insulin habit {summary['insulin_habit']:.0%} | held back "
          f"{summary['held_back']:.0%}  ({time.time() - t:.0f}s)")
    return results, summary


# The teacher's own answers on the exams: the line a warm-started model should reach.
teacher_probe = {name: P.summarize_probe(items, P.score_probe(items, P.teacher_texts(items), reward=reward_ref))
                 for name, items in (("probe", probe_items), ("test_probe", test_probe_items))}
for name, s in teacher_probe.items():
    print(f"[teacher, {name}] reward {s['reward']:+.3f} | {s['pct_of_best']:5.1f}% of best | all-zero {s['all_zero']:.0%} | "
          f"held back {s['held_back']:.0%}")


## 4. Before training: the untrained model

Phi-4-mini runs here with its LoRA still at zero, so it is the base model. These runs give the "before" numbers.
They are made now, because switching the adapter off later is not reliable with Unsloth's fused kernels.

In [ ]:
base_probe, base_probe_summary = run_probe(probe_items, "base, probe seeds")
base_test_probe, base_test_probe_summary = run_probe(test_probe_items, "base, test seeds")

In [ ]:
eval_seeds = list(TEST_SEEDS[:EVAL_SEEDS])
check_seeds = list(TEST_SEEDS[:CHECK_SEEDS])

t0 = time.time()
base_eps = EP.run_llm_episodes(TASKS, eval_seeds, generate, news=2, scenario="random", agent="llm", model="phi4-mini-base")
minutes = (time.time() - t0) / 60
per_answer = minutes * 60 / (len(base_eps) * 60)
print(f"base, random scenario: {len(base_eps)} episodes in {minutes:.0f} min ({per_answer:.2f}s per answer)")
print(f"estimate for the rest: post-training episodes ~{minutes * (1 + 2 * CHECK_SEEDS / EVAL_SEEDS):.0f} min")

base_fixed = EP.run_llm_episodes(TASKS, check_seeds, generate, news=2, scenario="fixed", agent="llm", model="phi4-mini-base")
E.write_jsonl(OUT / "base_random_news2.jsonl", base_eps)
E.write_jsonl(OUT / "base_fixed_news2.jsonl", base_fixed)
json.dump({"probe": base_probe_summary, "test_probe": base_test_probe_summary}, open(OUT / "base_probe.json", "w"), indent=1)

In [ ]:
# The references (rule agents, no model, a few seconds):
refs = {
    "baseline": [E.run_episode("baseline", t, s, news=2, scenario="random") for t in TASKS for s in eval_seeds],
    "plus3_everywhere": EP.run_rule_episodes(EP.FlatExtraAgent(3), "plus3_everywhere", TASKS, eval_seeds),
    "perfect_news": [E.run_episode("baseline_news", t, s, news=2, scenario="random") for t in TASKS for s in eval_seeds],
    "news_plan": [E.run_episode("news_plan", t, s, news=2, scenario="random") for t in TASKS for s in eval_seeds],
    "teacher": EP.run_rule_episodes(TEACH.TeacherAgent(), "teacher", TASKS, eval_seeds),
    "oracle": [E.run_episode("oracle", t, s, news=2, scenario="random") for t in TASKS for s in eval_seeds],
}
for name, recs in refs.items():
    print(f"{name:18s} {E.summarize(recs)['ALL']['mean']:.4f}")
print(f"{'base phi4-mini':18s} {E.summarize(base_eps)['ALL']['mean']:.4f}")

## 4b. Warm start: learn the teacher's answers (supervised)

`training/teacher.py` plays train-seed episodes and saves, for every news day (and a small share of the days without
news), the exact prompt the model gets and the teacher's answer. The teacher is the news plan (`agents.news_plan_targets`)
turned into whole extra days: it knows the event behind each news item and sizes the stock to the delay or the wave.
About half of its answers on news days are all zeros, which teaches holding back as well as reacting.

The model is trained on the answer tokens only (the prompt is masked), one pass over `SFT_EXAMPLES` examples. The probe
exam afterwards shows how close it got to the teacher's own line from part 3. Skipped when `WARM_START = False`.


In [ ]:
sft_summary, sft_eps = {}, []
start_probe_summary = base_probe_summary        # where GRPO starts from (replaced below after a warm start)

if WARM_START:
    t0 = time.time()
    sft_seeds = sorted(random.Random("sft-seeds").sample(list(TRAIN_SEED_POOL), SFT_EPISODES_PER_TASK))
    sft_rows = TEACH.sft_examples(TASKS, sft_seeds)          # train seeds only (checked), mixed teacher/baseline states
    random.Random("sft-order").shuffle(sft_rows)
    sft_rows = sft_rows[:SFT_EXAMPLES]
    stats = TEACH.label_stats(sft_rows)
    print(f"{len(sft_rows)} examples in {time.time() - t0:.0f}s: {stats['with_news']} with news "
          f"({stats['all_zero_with_news']} of them answered with all zeros), by task {stats['by_task']}")
    print("values given:", stats["nonzero_values"])

    limit = MAX_PROMPT_TOKENS + MAX_COMPLETION_TOKENS
    sft_encoded = [TEACH.encode_example(tokenizer, r["messages"], limit) for r in sft_rows]
    sizes = [len(e["input_ids"]) for e in sft_encoded]
    answer_sizes = [sum(l != TEACH.IGNORE_INDEX for l in e["labels"]) for e in sft_encoded]
    print(f"tokens per example: mean {st.mean(sizes):.0f}, max {max(sizes)}; answer tokens: mean {st.mean(answer_sizes):.0f}")
    print("an answer as trained:", repr(tokenizer.decode([l for l in sft_encoded[0]["labels"] if l != TEACH.IGNORE_INDEX])))


In [ ]:
if WARM_START:
    class EncodedExamples(torch.utils.data.Dataset):
        def __len__(self):
            return len(sft_encoded)

        def __getitem__(self, i):
            return sft_encoded[i]

    def one_example(batch):
        # Batch size 1: no padding, so the left-padding tokenizer setting does not matter here.
        assert len(batch) == 1
        return {k: torch.tensor([v]) for k, v in batch[0].items()}

    sft_args = TrainingArguments(
        output_dir=str(OUT / "sft_checkpoints"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=SFT_ACCUM,
        num_train_epochs=1,
        learning_rate=SFT_LEARNING_RATE,
        lr_scheduler_type="linear",
        warmup_steps=10,
        weight_decay=0.0,
        max_grad_norm=1.0,
        optim="adamw_8bit",
        logging_steps=10,
        save_strategy="no",
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
        remove_unused_columns=False,
        report_to="none",
        seed=0,
    )
    sft_trainer = Trainer(model=model, args=sft_args, train_dataset=EncodedExamples(), data_collator=one_example)
    t0 = time.time()
    back_to_training()
    sft_result = sft_trainer.train()
    sft_minutes = (time.time() - t0) / 60
    losses = [h["loss"] for h in sft_trainer.state.log_history if "loss" in h]
    print(f"warm start took {sft_minutes:.0f} min; loss {losses[0]:.3f} -> {losses[-1]:.3f}")

    SFT_ADAPTER_DIR = OUT / "phi4mini_pharma_lora_sft"       # saved before GRPO, in case the session ends
    model.save_pretrained(str(SFT_ADAPTER_DIR))
    tokenizer.save_pretrained(str(SFT_ADAPTER_DIR))


In [ ]:
if WARM_START:
    sft_probe, sft_probe_summary = run_probe(probe_items, "after warm start, probe seeds")
    sft_test_probe, sft_test_probe_summary = run_probe(test_probe_items, "after warm start, test seeds")
    start_probe_summary = sft_probe_summary
    d = P.compare_probes(sft_test_probe, base_test_probe)
    print(f"test probe, after warm start - base: {d['mean']:+.3f} points ± {d['ci95']:.3f} "
          f"(teacher's own answers: {teacher_probe['test_probe']['reward']:+.3f})")
    sft_summary = {"examples": len(sft_rows), "minutes": sft_minutes, "loss_first": losses[0], "loss_last": losses[-1],
                   "probe": sft_probe_summary, "test_probe": sft_test_probe_summary, "test_probe_vs_base": d}
    if EVAL_AFTER_SFT:
        sft_eps = EP.run_llm_episodes(TASKS, eval_seeds, generate, news=2, scenario="random", agent="llm",
                                      model="phi4-mini-sft")
        E.write_jsonl(OUT / "sft_random_news2.jsonl", sft_eps)
        print(f"after warm start, full episodes: {E.summarize(sft_eps)['ALL']['mean']:.4f} "
              f"(base {E.summarize(base_eps)['ALL']['mean']:.4f}, teacher {E.summarize(refs['teacher'])['ALL']['mean']:.4f})")
    json.dump(sft_summary, open(OUT / "sft_summary.json", "w"), indent=1, default=float)
    back_to_training()


## 5. Reward and training-health monitoring

`MonitoredReward` gives exactly the GRPO reward and also records health numbers. These are logged every step
next to TRL's own `reward`, `reward_std`, `kl`, `completion_length` and `frac_reward_zero_std`.

| Metric | Healthy | Worrying |
|---|---|---|
| `reward_ma20` (moving average) | rising slowly | flat for 100+ steps |
| `health/zero_std_groups` | below ~0.5 | near 1: groups teach nothing (raise the temperature) |
| `health/all_zero` | drifting to ~0.3–0.5 | 1.0: collapsed to "always 0" |
| `health/no_json` | goes to ~0 | rising |
| `kl` | small, slowly growing | jumps: lower `LEARNING_RATE` |
| `completion_length` | short, stable | growing: rambling |
| `health/right_product` | rising | falling |

In [ ]:
reward_fn = MonitoredReward(continuation=PLAYER)


class HealthCallback(TrainerCallback):
    """Adds the health numbers and a 20-step moving-average reward to every log, and keeps them all."""

    def __init__(self):
        self.recent = deque(maxlen=20)
        self.history = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs and "reward" not in logs:
            return
        logs.update(reward_fn.pop_stats())
        if "reward" in logs:
            self.recent.append(logs["reward"])
            logs["reward_ma20"] = sum(self.recent) / len(self.recent)
        row = {"step": state.global_step, **{k: v for k, v in logs.items() if isinstance(v, (int, float))}}
        self.history.append(row)
        if state.global_step % 5 == 0:
            keys = ["reward", "reward_ma20", "kl", "completion_length", "health/zero_std_groups",
                    "health/all_zero", "health/no_json", "health/right_product"]
            print(f"step {state.global_step:4d} | " + " | ".join(
                f"{k.split('/')[-1]} {row[k]:.3f}" for k in keys if k in row and row[k] == row[k]))
        with open(OUT / "train_log.jsonl", "a") as f:
            f.write(json.dumps(row) + "\n")


class ProbeCallback(TrainerCallback):
    """The probe exam (held-out seeds 9000-9099, greedy) every PROBE_EVERY steps."""

    def __init__(self):
        self.history = [{"step": 0, **start_probe_summary}]   # where GRPO starts: after the warm start, if any

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % PROBE_EVERY:
            return
        _, summary = run_probe(probe_items, f"step {state.global_step}")
        self.history.append({"step": state.global_step, **summary})
        json.dump(self.history, open(OUT / "probe_history.json", "w"), indent=1)
        back_to_training()


health_cb, probe_cb = HealthCallback(), ProbeCallback()

## 6. GRPO training

The settings are set by name, and any your TRL version does not know is reported and skipped. With
`scale_rewards=False` (Dr. GRPO) the advantages stay in reward points, so a group where every answer is within
0.01 points gives almost no push.

If you run out of memory, set `PER_DEVICE_BATCH = 4` and `GRAD_ACCUM = 4`.

In [ ]:
known = {f.name for f in fields(GRPOConfig)}
settings = dict(
    output_dir=str(OUT / "checkpoints"),
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=10,
    weight_decay=0.0,
    max_grad_norm=0.2,
    optim="adamw_8bit",
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_TOKENS,
    max_completion_length=MAX_COMPLETION_TOKENS,
    temperature=TEMPERATURE,
    beta=BETA_KL,
    scale_rewards=False,           # Dr. GRPO: no division by the group's std
    loss_type="dr_grpo",
    max_steps=TRAIN_STEPS,
    logging_steps=1,
    save_steps=SAVE_EVERY,
    save_total_limit=2,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    report_to="none",
    seed=0,
    log_completions=False,
)
skipped = sorted(set(settings) - known)
if skipped:
    print("not in this TRL version, skipped:", skipped)
config = GRPOConfig(**{k: v for k, v in settings.items() if k in known})

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=config,
    train_dataset=train_dataset,
    callbacks=[health_cb, probe_cb],
)

In [ ]:
t0 = time.time()
back_to_training()
trainer.train()
print(f"training took {(time.time() - t0) / 3600:.1f} h")

In [ ]:
# Training curves
import matplotlib.pyplot as plt

h = health_cb.history
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
panels = [("reward_ma20", "reward (20-step avg)"), ("kl", "KL to the base model"), ("completion_length", "completion length"),
          ("health/zero_std_groups", "groups with no spread"), ("health/all_zero", "answers all 0"),
          ("health/right_product", "largest extra on a news SKU")]
for ax, (key, title) in zip(axes.flat, panels):
    pts = [(r["step"], r[key]) for r in h if key in r and r[key] == r[key]]
    if pts:
        ax.plot(*zip(*pts))
    ax.set_title(title); ax.set_xlabel("step")
plt.tight_layout(); plt.savefig(OUT / "training_curves.png", dpi=120); plt.show()

ph = probe_cb.history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r["step"] for r in ph], [r["pct_of_best"] for r in ph], marker="o"); ax[0].set_title("probe: % of best achievable")
for key in ("right_product", "insulin_habit", "held_back", "all_zero"):
    ax[1].plot([r["step"] for r in ph], [r[key] for r in ph], marker="o", label=key)
ax[1].legend(); ax[1].set_title("probe: behaviour")
plt.tight_layout(); plt.savefig(OUT / "probe_curves.png", dpi=120); plt.show()

## 7. Save the LoRA adapter (before the long evaluation, in case the session ends)

In [ ]:
ADAPTER_DIR = OUT / "phi4mini_pharma_lora"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("saved", ADAPTER_DIR)

## 8. After training: did it improve?

1. **Test-seed probe exam:** the same 100 test news days as before training, base vs trained, paired.
2. **Full 60-day episodes:** test seeds, random scenario, news 2. Compared with the base model and four references:
   - the baseline;
   - "+3 everywhere";
   - perfect news reading (`baseline_news`, the formula from the exact events);
   - the teacher (the news plan answering in the model's format: what the warm start aims for);
   - the oracle.

   Paired by seed, with 95% intervals.
3. **Checks:**
   - **news 0:** with no news, the model should play like the baseline;
   - **fixed scenario:** it should do no worse than the base model.

**Success** = trained beats base with the interval above 0, and trained beats "+3 everywhere".

In [ ]:
trained_test_probe, trained_test_probe_summary = run_probe(test_probe_items, "trained, test seeds")
probe_diff = P.compare_probes(trained_test_probe, base_test_probe)
print(f"\ntest probe, trained - base: {probe_diff['mean']:+.3f} points ± {probe_diff['ci95']:.3f} "
      f"(better on {probe_diff['better']}, worse on {probe_diff['worse']} of {probe_diff['n']})")

In [ ]:
t0 = time.time()
trained_eps = EP.run_llm_episodes(TASKS, eval_seeds, generate, news=2, scenario="random", agent="llm",
                                  model="phi4-mini-grpo", save_replies=True)
trained_fixed = EP.run_llm_episodes(TASKS, check_seeds, generate, news=2, scenario="fixed", agent="llm", model="phi4-mini-grpo")
trained_news0 = EP.run_llm_episodes(TASKS, check_seeds, generate, news=0, scenario="random", agent="llm", model="phi4-mini-grpo")
print(f"episodes took {(time.time() - t0) / 60:.0f} min")
E.write_jsonl(OUT / "trained_random_news2.jsonl", trained_eps)
E.write_jsonl(OUT / "trained_fixed_news2.jsonl", trained_fixed)
E.write_jsonl(OUT / "trained_random_news0.jsonl", trained_news0)

In [ ]:
def line(name, comp):
    a = comp["ALL"]
    gap = f" | closes {a['gap_closed']:.0%} of the gap to the oracle" if "gap_closed" in a else ""
    return f"{name:34s} {a['mean']:+.4f} ± {a['ci95']:.4f}{gap}"


scores = {"trained phi4-mini": trained_eps, **({"after warm start": sft_eps} if sft_eps else {}),
          "base phi4-mini": base_eps, **refs}
print("Mean score, random scenario, news 2, test seeds:")
for name, recs in scores.items():
    print(f"  {name:20s} {E.summarize(recs)['ALL']['mean']:.4f}")

print("\nPaired differences (per seed):")
vs_base = E.paired_comparison(trained_eps, base_eps, refs["oracle"])
vs_plus3 = E.paired_comparison(trained_eps, refs["plus3_everywhere"])
print(line("trained - base", vs_base))
print(line("trained - +3 everywhere", vs_plus3))
print(line("trained - baseline", E.paired_comparison(trained_eps, refs["baseline"], refs["oracle"])))
print(line("trained - perfect news reading", E.paired_comparison(trained_eps, refs["perfect_news"])))
vs_teacher = E.paired_comparison(trained_eps, refs["teacher"])
print(line("trained - teacher", vs_teacher))
if sft_eps:
    print(line("after warm start - base", E.paired_comparison(sft_eps, base_eps, refs["oracle"])))
    print(line("after warm start - teacher", E.paired_comparison(sft_eps, refs["teacher"])))
    print(line("GRPO's part: trained - after warm start", E.paired_comparison(trained_eps, sft_eps, refs["oracle"])))
print(line("base - baseline", E.paired_comparison(base_eps, refs["baseline"], refs["oracle"])))
print("\nper task, trained - base:")
for task, s in vs_base.items():
    if task != "ALL":
        print(f"  {task:22s} {s['mean']:+.4f} ± {s['ci95']:.4f}")

In [ ]:
baseline_news0 = [E.run_episode("baseline", t, s, news=0, scenario="random") for t in TASKS for s in check_seeds]
news0 = E.paired_comparison(trained_news0, baseline_news0)["ALL"]
fixed = E.paired_comparison(trained_fixed, base_fixed)["ALL"]
print(line("news 0: trained - baseline", {"ALL": news0}), " (should be close to 0: no news, nothing to add)")
print(line("fixed scenario: trained - base", {"ALL": fixed}), " (should not be clearly below 0)")
print(f"parse failures per episode, trained: {st.mean(r['parse_failures'] for r in trained_eps):.1f}, "
      f"base: {st.mean(r['parse_failures'] for r in base_eps):.1f}")

a, b = vs_base["ALL"], vs_plus3["ALL"]
signal = a["mean"] - a["ci95"] > 0
beats_plus3 = b["mean"] > 0
print("\n==== VERDICT ====")
print(f"trained beats base (interval above 0): {'YES' if signal else 'no'}  ({a['mean']:+.4f} ± {a['ci95']:.4f})")
print(f"trained beats +3 everywhere:           {'YES' if beats_plus3 else 'no'}  ({b['mean']:+.4f} ± {b['ci95']:.4f})")
print(f"test probe, trained - base:            {probe_diff['mean']:+.3f} ± {probe_diff['ci95']:.3f} points")
t = vs_teacher["ALL"]
print(f"trained - teacher (the warm start's aim): {t['mean']:+.4f} ± {t['ci95']:.4f}")
if signal and beats_plus3:
    print("-> first positive signal: the model learned to use the news.")
elif signal:
    print("-> better than the base model, but not yet better than simply adding 3 days everywhere.")
else:
    print("-> no clear signal yet. Check the probe curve and the health metrics (part 6) to see why.")

summary = {
    "config": {k: v for k, v in settings.items() if isinstance(v, (int, float, str, bool))},
    "mean_scores": {n: E.summarize(r)["ALL"]["mean"] for n, r in scores.items()},
    "trained_vs_base": vs_base, "trained_vs_plus3": vs_plus3, "news0_vs_baseline": news0,
    "fixed_vs_base": fixed, "test_probe_base": base_test_probe_summary,
    "test_probe_trained": trained_test_probe_summary, "test_probe_diff": probe_diff,
    "probe_history": probe_cb.history,
    "warm_start": WARM_START, "player": PLAYER, "trained_vs_teacher": vs_teacher,
    "teacher_probe": teacher_probe, "sft": sft_summary,
}
json.dump(summary, open(OUT / "summary.json", "w"), indent=1, default=float)

## 9. A look at the answers, GGUF export, results zip

In [ ]:
# The probe days where training changed the reward the most (test seeds)
changes = sorted(zip(test_probe_items, base_test_probe, trained_test_probe),
                 key=lambda x: abs(x[2]["reward"] - x[1]["reward"]), reverse=True)
for item, b, t in changes[:6]:
    news = " / ".join(build_state(item.prompt)[1].news)
    print(f"--- {item.prompt.task}, seed {item.prompt.seed}, day {item.prompt.day} | news SKUs {item.skus} | "
          f"best reference {item.best:+.2f}")
    print("  news:", news[:300])
    print(f"  base    {b['reward']:+.2f}  {b['text'][:150]!r}")
    print(f"  trained {t['reward']:+.2f}  {t['text'][:150]!r}")

In [ ]:
# GGUF for Ollama (about 10-20 min: builds llama.cpp and quantises). Then, on your computer:
#   ollama create pharma-phi4-mini-grpo -f Modelfile        (Modelfile: FROM ./<the .gguf file>, PARAMETER num_ctx 8192)
#   uv run python local_llm/run_local_llm.py --server-model pharma-phi4-mini-grpo --news 2 --random-scenarios
if EXPORT_GGUF:
    try:
        model.save_pretrained_gguf(str(OUT / "phi4mini_pharma_gguf"), tokenizer, quantization_method="q4_k_m")
        gguf = sorted((OUT / "phi4mini_pharma_gguf").glob("*.gguf"))
        print("GGUF:", gguf)
        if gguf:
            (OUT / "phi4mini_pharma_gguf" / "Modelfile").write_text(f"FROM ./{gguf[-1].name}\nPARAMETER num_ctx 8192\n")
    except Exception as exc:
        print("GGUF export failed (the LoRA adapter is saved; it can be exported later):", exc)

In [ ]:
import shutil

# Logs, results and the LoRA adapter (not the checkpoints or the GGUF) in one zip to download
bundle = Path(WORK) / "grpo_results"
bundle.mkdir(exist_ok=True)
for f in OUT.iterdir():
    if f.is_file():
        shutil.copy(f, bundle / f.name)
shutil.copytree(ADAPTER_DIR, bundle / ADAPTER_DIR.name, dirs_exist_ok=True)
print(shutil.make_archive(str(bundle), "zip", root_dir=bundle))